In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Live_Query_Engine_Test") \
    .master("local[2]") \
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1,"
            "org.apache.hadoop:hadoop-aws:3.4.1,"
            "software.amazon.awssdk:bundle:2.29.38") \
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local",           "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type",      "rest") \
    .config("spark.sql.catalog.local.uri",       "http://rest-catalog:8181") \
    .config("spark.sql.catalog.local.warehouse", "s3a://business-data/") \
    .config("spark.sql.catalog.local.s3.endpoint",          "http://minio:9000") \
    .config("spark.sql.catalog.local.s3.path-style-access", "true") \
    .config("spark.sql.catalog.local.s3.region",            "us-east-1") \
    .config("spark.hadoop.fs.s3a.endpoint",          "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key",        "admin") \
    .config("spark.hadoop.fs.s3a.secret.key",        "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print(f"Spark Session Ready! Version: {spark.version}")

Spark Session Ready! Version: 4.0.2


In [2]:
# Query the live Iceberg table
spark.sql("""
    SELECT 
        traffic_event_ts, 
        traffic_borough, 
        traffic_speed, 
        aq_pm25_ugm3, 
        weather_temperature
    FROM local.db.enriched_traffic
    ORDER BY traffic_event_ts DESC
    LIMIT 10
""").show(truncate=False)

+-------------------+---------------+-------------+------------+-------------------+
|traffic_event_ts   |traffic_borough|traffic_speed|aq_pm25_ugm3|weather_temperature|
+-------------------+---------------+-------------+------------+-------------------+
|2026-05-01 01:56:13|Manhattan      |4.34         |2.0         |54                 |
|2026-05-01 01:56:13|Manhattan      |4.34         |1.8         |54                 |
|2026-05-01 01:56:13|Manhattan      |4.34         |1.8         |54                 |
|2026-05-01 01:56:13|Manhattan      |4.34         |1.8         |52                 |
|2026-05-01 01:56:13|Manhattan      |4.34         |2.0         |54                 |
|2026-05-01 01:56:13|Manhattan      |4.34         |1.8         |54                 |
|2026-05-01 01:56:13|Manhattan      |4.34         |1.8         |52                 |
|2026-05-01 01:56:13|Manhattan      |4.34         |1.8         |54                 |
|2026-05-01 01:56:13|Manhattan      |4.34         |2.0         |5

In [3]:
# Verify the aggregations and non-null air quality readings
spark.sql("""
    SELECT 
        traffic_borough, 
        COUNT(*) as total_records,
        ROUND(AVG(traffic_speed), 2) as avg_speed_mph, 
        ROUND(AVG(aq_pm25_ugm3), 2) as avg_pm25,
        MAX(weather_temperature) as current_temp
    FROM local.db.enriched_traffic
    GROUP BY traffic_borough
    ORDER BY avg_speed_mph ASC
""").show(truncate=False)

+---------------+-------------+-------------+--------+------------+
|traffic_borough|total_records|avg_speed_mph|avg_pm25|current_temp|
+---------------+-------------+-------------+--------+------------+
|Manhattan      |1202520      |13.07        |4.08    |59          |
|Bronx          |926050       |17.2         |17.6    |59          |
|Queens         |612430       |22.18        |0.0     |59          |
|Staten Island  |454380       |34.1         |1.31    |59          |
|Brooklyn       |1034230      |41.27        |0.64    |59          |
+---------------+-------------+-------------+--------+------------+



In [4]:
# 1. Read the raw Parquet dumps directly from MinIO
raw_traffic = spark.read.parquet("s3a://raw-data/traffic/")
raw_traffic.createOrReplaceTempView("raw_traffic_view")

# 2. See how many records have successfully landed
spark.sql("""
    SELECT borough, COUNT(*) as rows_ingested 
    FROM raw_traffic_view 
    GROUP BY borough
""").show()

+-------------+-------------+
|      borough|rows_ingested|
+-------------+-------------+
|       Queens|         2236|
|     Brooklyn|          726|
|Staten Island|         1561|
|    Manhattan|         1571|
|        Bronx|         1443|
+-------------+-------------+



In [5]:
spark.read.parquet("s3a://raw-data/weather/").selectExpr(
    "MIN(kafka_timestamp) as oldest", "MAX(kafka_timestamp) as newest"
).show(truncate=False)

spark.read.parquet("s3a://raw-data/openaq/").selectExpr(
    "MIN(kafka_timestamp) as oldest", "MAX(kafka_timestamp) as newest"
).show(truncate=False)

+-----------------------+-----------------------+
|oldest                 |newest                 |
+-----------------------+-----------------------+
|2026-04-30 23:05:27.324|2026-05-01 03:43:05.769|
+-----------------------+-----------------------+

+-----------------------+-----------------------+
|oldest                 |newest                 |
+-----------------------+-----------------------+
|2026-04-30 23:05:26.624|2026-05-01 03:43:45.325|
+-----------------------+-----------------------+



In [6]:
spark.sql("""
    SELECT MIN(traffic_event_ts), MAX(traffic_event_ts), COUNT(*) as total
    FROM local.db.enriched_traffic
""").show(truncate=False)

+---------------------+---------------------+-------+
|min(traffic_event_ts)|max(traffic_event_ts)|total  |
+---------------------+---------------------+-------+
|2026-04-30 21:13:00  |2026-05-01 01:56:13  |4229610|
+---------------------+---------------------+-------+



In [7]:
spark.sql("""
    SELECT 
        MAX(traffic_event_ts) as latest_traffic,
        (SELECT MIN(kafka_timestamp) FROM parquet.`s3a://raw-data/openaq/`) as earliest_aq
    FROM local.db.enriched_traffic
""").show(truncate=False)

+-------------------+-----------------------+
|latest_traffic     |earliest_aq            |
+-------------------+-----------------------+
|2026-05-01 01:56:13|2026-04-30 23:05:26.624|
+-------------------+-----------------------+



In [8]:
spark.sql("""
    SELECT 
        traffic_borough,
        traffic_id,
        COUNT(*) as aq_matches_per_traffic_row
    FROM local.db.enriched_traffic
    GROUP BY traffic_borough, traffic_id
    ORDER BY aq_matches_per_traffic_row DESC
    LIMIT 20
""").show(truncate=False)

+---------------+----------+--------------------------+
|traffic_borough|traffic_id|aq_matches_per_traffic_row|
+---------------+----------+--------------------------+
|Manhattan      |124       |558420                    |
|Brooklyn       |261       |435840                    |
|Brooklyn       |258       |422220                    |
|Manhattan      |150       |251970                    |
|Queens         |365       |211110                    |
|Bronx          |191       |140160                    |
|Bronx          |195       |140160                    |
|Bronx          |177       |137240                    |
|Manhattan      |265       |134320                    |
|Bronx          |190       |134320                    |
|Bronx          |186       |128480                    |
|Staten Island  |385       |122580                    |
|Brooklyn       |154       |60040                     |
|Queens         |423       |34760                     |
|Queens         |318       |33180               

In [9]:
spark.sql("""
    SELECT 
        h3_index,
        COUNT(*) as rows
    FROM local.db.enriched_traffic
    GROUP BY h3_index
    ORDER BY rows DESC
    LIMIT 10
""").show(truncate=False)

+---------------+------+
|h3_index       |rows  |
+---------------+------+
|882a10755dfffff|858060|
|882a100ae3fffff|814680|
|882a107297fffff|558420|
|882a1072d7fffff|251970|
|882a100d01fffff|211110|
|882a106231fffff|122580|
|882a100f3dfffff|82950 |
|882a10729dfffff|70310 |
|882a1001e3fffff|70310 |
|882a100e2bfffff|67150 |
+---------------+------+

